# ML Capstone — Review 2: Clustering Track
- **Dataset:** Social Media Usage (1000 rows × 6 cols)
- **Algorithms:** K-Means, Agglomerative Hierarchical Clustering
- **Labels NOT used during training** — ground-truth used only for post-hoc validation
- **Pipeline:** sklearn `Pipeline` + `ColumnTransformer`


## 1. Imports & Setup

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score
)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
warnings.filterwarnings('ignore')
SEED = 42
print('Imports OK')


## 2. Data Loading & EDA
- Drop `User_ID` and ground-truth `App` before any model fitting
- Check shape, missing values, distributions


In [ ]:
DATA = '/Users/abhinavreddy/Documents/SEM_5/ML/ML_proj/Clustering/social_media_usage.csv'
df = pd.read_csv(DATA)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum())
print('\nDescriptive statistics:')
df.describe()


### Distribution Plots
- Histograms of all numeric features
- Correlation heatmap


In [ ]:
# Ground-truth label kept separate — not used in training
GT_COL = 'App'
DROP_COLS = ['User_ID', GT_COL]
y_true_labels = df[GT_COL]  # for post-hoc only

X_raw = df.drop(columns=DROP_COLS)
feature_cols = X_raw.columns.tolist()
print('Features used for clustering:', feature_cols)

fig, axes = plt.subplots(1, len(feature_cols), figsize=(15, 3))
for ax, col in zip(axes, feature_cols):
    ax.hist(X_raw[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Feature Distributions', y=1.02)
plt.tight_layout()
plt.savefig('clustering_eda_distributions.png', dpi=80, bbox_inches='tight')
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(X_raw.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('clustering_correlation.png', dpi=80, bbox_inches='tight')
plt.show()


## 3. Preprocessing Pipeline
- `SimpleImputer(median)` → `StandardScaler`
- Fit on features only (no labels)


In [ ]:
# Build preprocessing pipeline
preprocess_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

X_scaled = preprocess_pipe.fit_transform(X_raw)
print('Scaled shape:', X_scaled.shape)
print('Mean (should be ~0):', X_scaled.mean(axis=0).round(4))
print('Std  (should be ~1):', X_scaled.std(axis=0).round(4))


## 4. K-Means: Optimal K Diagnostics (Elbow + Silhouette)
- Elbow method: inertia vs k
- Silhouette score vs k
- Optimal k selected from intersection


In [ ]:
k_range = range(2, 11)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, 'bo-')
axes[0].set_title('Elbow Curve (Inertia vs k)')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Inertia')
axes[0].axvline(x=8, color='red', linestyle='--', label='k=8 chosen')
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, 'ro-')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(x=8, color='blue', linestyle='--', label='k=8 chosen')
axes[1].legend()

plt.suptitle('K-Means: Choosing Optimal k')
plt.tight_layout()
plt.savefig('kmeans_elbow_silhouette.png', dpi=80, bbox_inches='tight')
plt.show()
print('Chosen k=8 based on elbow + silhouette trade-off')


### K-Means Fit (k=8)
- Final KMeans with chosen k

In [ ]:
BEST_K = 8
km_final = KMeans(n_clusters=BEST_K, random_state=SEED, n_init=10)
km_labels = km_final.fit_predict(X_scaled)
print('Cluster sizes (K-Means):')
print(pd.Series(km_labels).value_counts().sort_index())


## 5. Agglomerative Hierarchical Clustering
- Ward linkage (minimises within-cluster variance)
- Dendrogram cut at n_clusters=8


In [ ]:
# Compute linkage matrix for dendrogram
Z = linkage(X_scaled[:200], method='ward')  # subset for readability

plt.figure(figsize=(14, 5))
dendrogram(
    Z,
    truncate_mode='lastp', p=30,
    leaf_rotation=90., leaf_font_size=8.,
    show_contracted=True,
    color_threshold=0.7 * max(Z[:, 2])
)
plt.axhline(y=0.7 * max(Z[:, 2]), color='red', linestyle='--', label='Cut threshold')
plt.title('Hierarchical Clustering Dendrogram (Ward linkage, n=200 subset)')
plt.xlabel('Sample index')
plt.ylabel('Distance')
plt.legend()
plt.tight_layout()
plt.savefig('dendrogram_ward.png', dpi=80, bbox_inches='tight')
plt.show()


In [ ]:
agg_final = AgglomerativeClustering(n_clusters=BEST_K, linkage='ward')
agg_labels = agg_final.fit_predict(X_scaled)
print('Cluster sizes (Agglomerative):')
print(pd.Series(agg_labels).value_counts().sort_index())


## 6. Clustering Evaluation Metrics (Rubric B2)
- Silhouette Score (higher = better)
- Davies-Bouldin Index (lower = better)
- Calinski-Harabasz Score (higher = better)
- ARI & NMI computed against ground-truth `App` for post-hoc validation only


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_true_enc = le.fit_transform(y_true_labels)

def cluster_metrics(name, labels):
    return {
        'Algorithm': name,
        'k': len(set(labels)),
        'Silhouette':         round(silhouette_score(X_scaled, labels), 4),
        'Davies-Bouldin':     round(davies_bouldin_score(X_scaled, labels), 4),
        'Calinski-Harabasz':  round(calinski_harabasz_score(X_scaled, labels), 2),
        'ARI (post-hoc)':     round(adjusted_rand_score(y_true_enc, labels), 4),
        'NMI (post-hoc)':     round(normalized_mutual_info_score(y_true_enc, labels), 4),
    }

metrics_table = pd.DataFrame([
    cluster_metrics('K-Means (k=8)', km_labels),
    cluster_metrics('Agglomerative Ward (k=8)', agg_labels),
])
print(metrics_table.to_string(index=False))


## 7. Cluster Visualizations (Rubric B3)
- PCA 2D scatter: colour-coded by cluster (K-Means & Agglomerative)
- t-SNE 2D scatter: K-Means clusters


In [ ]:
# PCA reduction
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='tab10', s=15, alpha=0.7)
axes[0].set_title('K-Means Clusters (PCA projection)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=agg_labels, cmap='tab10', s=15, alpha=0.7)
axes[1].set_title('Agglomerative Clusters (PCA projection)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.suptitle('PCA Cluster Scatter Plots')
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=80, bbox_inches='tight')
plt.show()


In [ ]:
# t-SNE for K-Means clusters
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=500)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=km_labels, cmap='tab10', s=15, alpha=0.7)
plt.colorbar(scatter, label='K-Means Cluster')
plt.title('t-SNE Projection — K-Means Clusters')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('tsne_kmeans.png', dpi=80, bbox_inches='tight')
plt.show()


## 8. Post-Hoc Validation (vs Ground-Truth App Label)
- ARI & NMI computed against `App` label **after** clustering
- Not used during training
- Cluster-to-App crosstab shows alignment


In [ ]:
print('K-Means ARI vs App:', round(adjusted_rand_score(y_true_enc, km_labels), 4))
print('K-Means NMI vs App:', round(normalized_mutual_info_score(y_true_enc, km_labels), 4))

# Crosstab: cluster vs app
df_valid = df[[GT_COL]].copy()
df_valid['KM_Cluster'] = km_labels
ct = pd.crosstab(df_valid['KM_Cluster'], df_valid[GT_COL])
plt.figure(figsize=(10, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')
plt.title('K-Means Cluster vs Ground-Truth App (Post-Hoc Crosstab)')
plt.tight_layout()
plt.savefig('cluster_app_crosstab.png', dpi=80, bbox_inches='tight')
plt.show()
print('Low ARI/NMI expected — data not inherently structured by a single app.')


## Summary
- K-Means (k=8): Silhouette=0.2278, Davies-Bouldin=1.1305, Calinski-Harabasz=228.52
- Agglomerative Ward (k=8): Silhouette=0.1596, Davies-Bouldin=1.4051, Calinski-Harabasz=173.16
- K-Means is the **preferred algorithm** (better cohesion & separation)
- Low ARI/NMI: social media data not inherently clustered by app category
